# Test Backwards DeepFOC

**Table of contents**<a id='toc0_'></a>    
- 1. [Imports](#toc1_)    
- 2. [Setings](#toc2_)    
- 3. [Simultaneous solve](#toc3_)    
- 4. [Backwards refinement](#toc4_)    
- 5. [Pure backwards](#toc5_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=2
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## 1. <a id='toc1_'></a>[Imports](#toc0_)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import time
import numpy as np
import torch

In [3]:
import matplotlib.pyplot as plt
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.rcParams.update({'axes.grid':True,'grid.color':'black','grid.alpha':'0.25','grid.linestyle':'--'})
plt.rcParams.update({'font.size': 14})

In [4]:
from EconDLSolvers import choose_gpu, clean_solving_json
from BufferStockModel import BufferStockModelClass
clean_solving_json()

## 2. <a id='toc2_'></a>[Setings](#toc0_)

In [5]:
LOAD_SIMULT = False
K_time_simult = 1.0
K_time = 1.0
par = {'Nstates_fixed':0}

In [6]:
algoname = 'DeepFOC'
filename = '../output/BufferStockModel_DeepFOC_3D.pt'

In [7]:
def print_R(model):
    model.simulate_R()
    R = model.sim.R.item()
    print(f'{R = :12.8f}')

## 3. <a id='toc3_'></a>[Simultaneous solve](#toc0_)

In [8]:
device = choose_gpu()

GPU 0: 44.40GB free [NVIDIA L40]
Best GPU: 0


In [9]:
if LOAD_SIMULT:
   
    model_simult = BufferStockModelClass(load=filename,device=device)

else:

    model_simult = BufferStockModelClass(
        algoname=algoname,device=device,
        par=par,
        train={'K_time':K_time_simult})
    
    model_simult.solve(do_print=True,do_print_all=False)

started solving: 2026-01-30 22:29:35


k =     0 of inf: sim.R =   -34.37154388 [best:   -34.37154388] [3.3 secs] [value_epochs =   0] [policy_epochs =  15] [  0.06 mins] [policy_lr = 1.0e-03]


k =    10 of inf: sim.R =     2.02215052 [best:     2.02215052] [3.1 secs] [value_epochs =   0] [policy_epochs =  15] [  0.15 mins] [policy_lr = 1.0e-03]


k =    20 of inf: sim.R =     2.04579163 [best:     2.04579163] [3.1 secs] [value_epochs =   0] [policy_epochs =  15] [  0.24 mins] [policy_lr = 1.0e-03]


k =    30 of inf: sim.R =     2.05556679 [best:     2.05556679] [3.1 secs] [value_epochs =   0] [policy_epochs =  15] [  0.33 mins] [policy_lr = 1.0e-03]


...

k =   100 of inf: sim.R =     2.05531240 [best:     2.06358933] [0.5 secs] [value_epochs =   0] [policy_epochs =  15] [  0.82 mins] [policy_lr = 9.9e-04] no improvements in 0.1 mins


k =   110 of inf: sim.R =     2.06336451 [best:     2.06358933] [0.5 secs] [value_epochs =   0] [policy_epochs =  15] [  0.87 mins] [policy_lr = 9.9e-04] no improvements in 0.2 mins


k =   120 of inf: sim.R =     2.06397271 [best:     2.06397271] [3.2 secs] [value_epochs =   0] [policy_epochs =  15] [  0.97 mins] [policy_lr = 9.9e-04]


Terminating after 129 episodes, max time 1.0 mins reached


R = 2.0640, time = 1.0 mins, iter = 129, policy epochs = 14.09, value epochs = 0.00
Simulating multiple Rs


In [10]:
print_R(model_simult)

R =   2.06397271


## 4. <a id='toc4_'></a>[Backwards refinement](#toc0_)

In [11]:
train = {}
train['Nneurons_policy_t'] = np.array([50,50])
train['N'] = 1_000_000
train['batch_size'] = 100_000
train['K_time'] = K_time

In [12]:
model_refine = BufferStockModelClass(
    algoname=f'{algoname}Backward',device=device,
    par=par,train=train
)

In [13]:
model_refine.solve(do_print=True,model_simult=model_simult)
print(f'{model_refine.info["time"]/60:.1f} mins')

started solving: 2026-01-30 22:30:41


t =  53
 epoch =     0: 5.5e-05

 epoch =    10: 5.2e-05 time limit reached [  1.2 secs]
t =  52


 epoch =     0: 2.7e-05 time limit reached [  2.1 secs]
t =  51


 epoch =     0: 1.2e-05 time limit reached [  2.1 secs]
t =  50


...

 epoch =     0: 9.3e-06 time limit reached [  2.1 secs]
t =   1


 epoch =     0: 8.5e-06 time limit reached [  2.1 secs]
t =   0


 epoch =     0: 6.3e-06 time limit reached [  2.1 secs]


Simulating multiple Rs


4.0 mins


R:

In [14]:
print_R(model_simult)
print_R(model_refine)

R =   2.06397271


R =   2.06399083


In [15]:
del model_refine

## 5. <a id='toc5_'></a>[Pure backwards](#toc0_)

In [16]:
train = {}
train['use_simult_in_backward'] = False
train['Nneurons_policy_t'] = np.array([50,50])
train['N'] = 1_000_000
train['batch_size'] = 100_000
train['K_time'] = K_time

In [17]:
model_pure = BufferStockModelClass(
    algoname=f'{algoname}Backward',device=device,
    par={'Nstates_fixed':0},train=train
)

In [18]:
model_pure.solve(do_print=True)
print(f'{model_pure.info["time"]/60:.1f} mins')

started solving: 2026-01-30 22:32:49


t =  53
 epoch =     0: 1.4e-01

 epoch =    10: 8.7e-03

 epoch =    20: 1.8e-03

 epoch =    30: 8.5e-04

...

 epoch =     6: 9.1e-06 time limit reached [  1.1 secs]
t =   1
 epoch =     0: 2.4e-04

 epoch =     6: 8.3e-06 time limit reached [  1.1 secs]
t =   0
 epoch =     0: 2.3e-04

 epoch =     6: 7.4e-06 time limit reached [  1.1 secs]


Simulating multiple Rs


2.2 mins


In [19]:
print_R(model_simult)
print_R(model_pure)

R =   2.06397271
R = -28.98239708


In [20]:
del model_pure